In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
ea = {"euler_angles": (30, -30, 0)}

In [ ]:
box = Box(p1=(0, 0, 0), p2=(10, 10, 20))
box.mat("box")
box.faces.name = "box"
box.faces.Min(X).name = "left"
box.faces.Max(X).name = "right"
box.faces.Min(Y).name = "bottom"
box.faces.Max(Y).name = "top"
box.faces.Min(Z).name = "rear"
box.faces.Max(Z).name = "front"
box.edges.name = "edges"

cyl = Cylinder(p=(5, 5, 0), d=Z, r=2, h=20)
cyl.mat("tube")
cyl.faces.name = "interface"
cyl.faces.Min(Z).name = "inlet"
cyl.faces.Max(Z).name = "outlet"

shape = box - cyl

ngmesh = OCCGeometry(shape).GenerateMesh(maxh=1)
mesh = Mesh(ngmesh)
mesh.Curve(3)
Draw(mesh, **ea);

In [ ]:
# Material Parameters
E = 21e+3
nu = 0.35
mu = E / 2 / (1 + nu)
lam = E * nu / ((1 + nu) * (1 - 2 * nu))

# Initialize FE Space
fes = VectorH1(mesh, order=2, dirichlet="bottom")

# Initialize symbolic u as trial function 
# for constructing symbolic term (such as NeoHooke)
u = fes.TrialFunction()

# Components for Strain Energy
I = Id(mesh.dim)
F = I + Grad(u)
C = F.trans * F
E = 0.5 * (C - I)

def Pow(a, b):
    return a**b  # exp (log(a)*b)

def NeoHooke(C):
    return 0.5 * mu * (Trace(C - I) + 2 * mu / lam * Pow(Det(C), -lam / 2 / mu) - 1)

# Constant Loading
# force = CoefficientFunction((0, 10, 0))

# Gravity Loading
# rho = 1100e-6
# g = 9.81e+3
# fgrav = -(rho * g)
# force = CoefficientFunction((0, fgrav, 0))

# Normal Loading
fnormal = 10
n = specialcf.normal(mesh.dim)

# Load Factor
factor = Parameter(0)

# Stiffness Matrices
a = BilinearForm(fes, symmetric=False)
a += Variation(NeoHooke(C).Compile() * dx)
a += Variation((-factor * -fnormal * InnerProduct(n, u)).Compile() * ds("interface"))

# Initialize solution u in a format of grid function
u = GridFunction(fes)
u.vec[:] = 0  # Define an initial guess (zeros)

# Allocate the memory for calculated components in NR
res = u.vec.CreateVector()  # residual
w = u.vec.CreateVector()    # incremental displacement solution (from NR)

In [ ]:
# Define solving parameters
n_loadstep = 10
n_nr = 5

# Solve with load stepping
for i_ls in range(1, n_loadstep):

    print(f"Load Step: {i_ls}/{n_loadstep}")
    factor.Set(i_ls/n_loadstep)

    for i_nr in range(n_nr):
        print(f"\tNewton iteration: {i_nr + 1}/{n_nr}")
        print(f"\t\tEnergy: {a.Energy(u.vec):.6f}")
        a.Apply(u.vec, res)
        a.AssembleLinearization(u.vec)
        inv = a.mat.Inverse(fes.FreeDofs())
        w.data = inv * res
        print(f"\t\tErr^2: {InnerProduct(w, res):.6e}")
        u.vec.data -= w

    Draw(u, mesh, deformation=True, scale=1e3, **ea)